# E016 — Audio FBank 40+Δ+ΔΔ (120 features/frame)

Replaces MFCC 13+Δ+ΔΔ (39 features) from E008 with log mel-filterbank
features (FBank 40+Δ+ΔΔ = 120 features/frame). DCT compression is skipped;
all 40 log-mel energies are kept. Everything else identical to E008 +All:
UBM 32, MAP r=16, noise SNR=20dB + speed ±10%, LOSO 3-fold, seed=67.

**Comparison target:** E008 +All: fold0=3.47, fold1=8.33, fold2=0.83, mean=4.21±3.11%

In [ ]:
from pathlib import Path
import sys, copy
sys.path.insert(0, str(Path("../src").resolve()))

import numpy as np
import librosa
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture
from sklearn.metrics import roc_curve, auc
from scipy.special import logsumexp
from scipy.stats import norm as scipy_norm
import pandas as pd

from data.splits import load_manifest, iter_folds_loso
from eval.metrics import compute_eer, compute_min_dcf

COLORS = {
    "target":    "#E74C3C",
    "nontarget": "#2E86AB",
    "green":     "#27AE60",
    "purple":    "#8E44AD",
    "gray":      "#95A5A6",
    "orange":    "#E67E22",
    "fbank":     "#8E44AD",
    "e008":      "#95A5A6",
}
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

DATA = Path("../data").resolve()
manifest = load_manifest(DATA)
y_all = manifest["label"].to_numpy()
SEED = 67
print(f"{len(manifest)} samples — {manifest.label.sum()} target, {(manifest.label==0).sum()} non-target")

## 1. Helper functions

All augmentation and UBM/MAP functions are identical to E008.
The only change is `extract_fbank` replacing `extract_mfcc`.

In [ ]:
def find_wav(stem: str, data_dir: Path) -> Path:
    for sf in ["target_train", "target_dev", "non_target_train", "non_target_dev"]:
        p = data_dir / sf / (stem + ".wav")
        if p.exists():
            return p
    raise FileNotFoundError(stem)


def aug_noise(y: np.ndarray, snr_db: float = 20.0, rng: np.random.Generator = None) -> np.ndarray:
    """Add white noise at target SNR."""
    signal_power = np.mean(y ** 2) + 1e-10
    noise_power  = signal_power / (10 ** (snr_db / 10))
    noise = rng.normal(0, np.sqrt(noise_power), len(y)).astype(y.dtype)
    return y + noise


def aug_speed(y: np.ndarray, rate_range=(0.9, 1.1), rng: np.random.Generator = None) -> np.ndarray:
    """Random time stretch without changing pitch."""
    rate = rng.uniform(*rate_range)
    return librosa.effects.time_stretch(y, rate=rate)


def extract_fbank(y: np.ndarray, sr: int, n_mels: int = 40) -> np.ndarray:
    """Log mel-filterbank + delta + delta-delta + CMN. Returns (T, 120)."""
    mel    = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels)
    fbank  = librosa.power_to_db(mel)                    # (40, T)
    delta  = librosa.feature.delta(fbank)
    delta2 = librosa.feature.delta(fbank, order=2)
    feat   = np.vstack([fbank, delta, delta2]).T          # (T, 120)
    feat  -= feat.mean(axis=0)                            # CMN
    return feat


def load_and_augment(wav_path: Path, rng: np.random.Generator):
    """Load WAV and return list of (y, sr) for +All config."""
    y, sr = librosa.load(wav_path, sr=None, mono=True)
    return [
        (y, sr),
        (aug_noise(y, snr_db=20.0, rng=rng), sr),
        (aug_speed(y, rng=rng), sr),
    ]


def extract_batch_aug(df: pd.DataFrame, data_dir: Path, seed: int):
    """Extract FBank frames for all samples with +All augmentation (train fold)."""
    rng = np.random.default_rng(seed)
    all_feat, all_labels = [], []
    for _, row in df.iterrows():
        for y_aug, sr in load_and_augment(find_wav(row["stem"], data_dir), rng):
            feat = extract_fbank(y_aug, sr)
            all_feat.append(feat)
            all_labels.extend([row["label"]] * len(feat))
    return np.vstack(all_feat), np.array(all_labels)


def train_ubm(X: np.ndarray, n_components: int = 32, seed: int = 67) -> GaussianMixture:
    return GaussianMixture(
        n_components=n_components, covariance_type="diag",
        max_iter=200, random_state=seed,
    ).fit(X)


def map_adapt(ubm: GaussianMixture, X_target: np.ndarray, r: float = 16.0) -> GaussianMixture:
    log_prob  = ubm._estimate_log_prob(X_target)
    log_resp  = log_prob + np.log(ubm.weights_)
    log_resp -= logsumexp(log_resp, axis=1, keepdims=True)
    resp      = np.exp(log_resp)
    n_k       = resp.sum(axis=0)
    mu_hat    = (resp.T @ X_target) / (n_k[:, None] + 1e-10)
    alpha     = n_k / (n_k + r)
    adapted   = copy.deepcopy(ubm)
    adapted.means_ = alpha[:, None] * mu_hat + (1 - alpha[:, None]) * ubm.means_
    return adapted


def score_wav(wav_path: Path, adapted: GaussianMixture, ubm: GaussianMixture) -> float:
    y, sr = librosa.load(wav_path, sr=None, mono=True)
    feat  = extract_fbank(y, sr)
    return float((adapted.score_samples(feat) - ubm.score_samples(feat)).mean())


print("All helpers defined. FBank dim = 120 (40 static + 40 delta + 40 delta2).")

## 2. FBank vs MFCC visualization

Compare the two feature representations on the same utterance to confirm
the FBank retains finer spectral structure that MFCC compresses away.

In [ ]:
def extract_mfcc_ref(y: np.ndarray, sr: int, n_mfcc: int = 13) -> np.ndarray:
    """MFCC reference (E008) — for visualization only."""
    mfcc   = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    delta  = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    mfcc   = np.vstack([mfcc, delta, delta2]).T
    mfcc  -= mfcc.mean(axis=0)
    return mfcc

sample_row = manifest[manifest.label == 1].iloc[0]
y_demo, sr_demo = librosa.load(find_wav(sample_row["stem"], DATA), sr=None, mono=True)

mfcc_demo  = extract_mfcc_ref(y_demo, sr_demo)
fbank_demo = extract_fbank(y_demo, sr_demo)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Static part
ax = axes[0, 0]
im = ax.imshow(mfcc_demo[:, :13].T, aspect="auto", origin="lower", cmap="magma")
ax.set_title("MFCC static 13 (E008)", fontsize=10)
ax.set_xlabel("Frame"); ax.set_ylabel("Coefficient")
plt.colorbar(im, ax=ax)

ax = axes[0, 1]
im = ax.imshow(fbank_demo[:, :40].T, aspect="auto", origin="lower", cmap="magma")
ax.set_title("FBank static 40 (E016)", fontsize=10)
ax.set_xlabel("Frame"); ax.set_ylabel("Mel bin")
plt.colorbar(im, ax=ax)

# Delta part
ax = axes[1, 0]
im = ax.imshow(mfcc_demo[:, 13:26].T, aspect="auto", origin="lower", cmap="RdBu_r")
ax.set_title("MFCC delta 13 (E008)", fontsize=10)
ax.set_xlabel("Frame"); ax.set_ylabel("Coefficient")
plt.colorbar(im, ax=ax)

ax = axes[1, 1]
im = ax.imshow(fbank_demo[:, 40:80].T, aspect="auto", origin="lower", cmap="RdBu_r")
ax.set_title("FBank delta 40 (E016)", fontsize=10)
ax.set_xlabel("Frame"); ax.set_ylabel("Mel bin")
plt.colorbar(im, ax=ax)

plt.suptitle(f"FBank vs MFCC — {sample_row['stem']} (target)",
             color=COLORS["target"], fontsize=12)
plt.tight_layout()
plt.show()

print(f"MFCC shape: {mfcc_demo.shape}  (T, 39)")
print(f"FBank shape: {fbank_demo.shape}  (T, 120)")

## 3. LOSO cross-validation with FBank

- UBM trained on non-target train-fold frames (FBank, +All aug)
- MAP adaptation on target train-fold frames
- Val utterances scored with original WAVs only (no augmentation)

In [ ]:
UBM_COMPONENTS = 32
MAP_R = 16.0

oof_scores   = np.full(len(manifest), np.nan)
fold_results = []

for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
    train_df = manifest.loc[train_idx]
    val_df   = manifest.loc[val_idx]

    print(f"\n--- Fold {fold_id} ---")
    print(f"  Train: {len(train_df)} utts  |  Val: {len(val_df)} utts")

    # Extract FBank features with +All augmentation (train fold only)
    X_train, y_train = extract_batch_aug(train_df, DATA, seed=SEED + fold_id)
    X_nt = X_train[y_train == 0]
    X_t  = X_train[y_train == 1]

    print(f"  Target frames: {len(X_t):,}  |  Non-target frames: {len(X_nt):,}  (aug)")

    # Train UBM on non-target frames
    ubm = train_ubm(X_nt, n_components=UBM_COMPONENTS, seed=SEED)
    print(f"  UBM trained ({UBM_COMPONENTS} components, FBank dim=120)")

    # MAP-adapt on target frames
    adapted = map_adapt(ubm, X_t, r=MAP_R)

    # Score val on ORIGINAL WAVs
    for idx, row in val_df.iterrows():
        oof_scores[idx] = score_wav(find_wav(row["stem"], DATA), adapted, ubm)

    val_scores = oof_scores[val_idx]
    val_labels = manifest.loc[val_idx, "label"].to_numpy()
    eer, _     = compute_eer(val_scores[val_labels==1], val_scores[val_labels==0])
    min_dcf, _ = compute_min_dcf(val_scores[val_labels==1], val_scores[val_labels==0])
    fold_results.append({"fold": fold_id, "eer": eer, "min_dcf": min_dcf})
    print(f"  → EER={eer*100:.2f}%,  min-DCF={min_dcf:.4f}")

print("\nAll folds done.")

## 4. Results table — FBank vs E008 baseline

In [ ]:
eers    = [r["eer"]*100  for r in fold_results]
dcfs    = [r["min_dcf"] for r in fold_results]
mean_e  = np.mean(eers)
std_e   = np.std(eers)
mean_d  = np.mean(dcfs)

E008_EERS = [3.47, 8.33, 0.83]
E008_DCFS = [0.0694, 0.0667, 0.0167]

print(f"{'':30} {'F0 EER':>8} {'F1 EER':>8} {'F2 EER':>8} {'Mean':>8} {'Std':>8} {'min-DCF':>9}")
print("-" * 82)
print(f"{'E008 +All (MFCC 13, 39-dim)':<30} "
      f"{E008_EERS[0]:>8.2f} {E008_EERS[1]:>8.2f} {E008_EERS[2]:>8.2f} "
      f"{np.mean(E008_EERS):>8.2f} {np.std(E008_EERS):>8.2f} {np.mean(E008_DCFS):>9.4f}")
print(f"{'E016 +All (FBank 40, 120-dim)':<30} "
      f"{eers[0]:>8.2f} {eers[1]:>8.2f} {eers[2]:>8.2f} "
      f"{mean_e:>8.2f} {std_e:>8.2f} {mean_d:>9.4f}")
print("-" * 82)

delta_mean = mean_e - np.mean(E008_EERS)
direction  = "improvement" if delta_mean < 0 else "regression"
print(f"\nDelta vs E008: {delta_mean:+.2f}% ({direction})")

# OOF overall
eer_oof, _   = compute_eer(oof_scores[y_all==1], oof_scores[y_all==0])
dcf_oof, thr = compute_min_dcf(oof_scores[y_all==1], oof_scores[y_all==0])
print(f"OOF overall: EER={eer_oof*100:.2f}%, min-DCF={dcf_oof:.4f}, threshold={thr:.3f}")

## 5. Visualizations

- Per-fold bar chart: FBank vs E008
- DET curve comparison
- Score distribution 2×2 (per fold + OOF)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Per-fold grouped bars
ax = axes[0]
x = np.arange(3)
width = 0.35
b1 = ax.bar(x - width/2, E008_EERS, width, label="E008 +All (MFCC 39d)",
            color=COLORS["e008"], alpha=0.85)
b2 = ax.bar(x + width/2, eers, width, label="E016 +All (FBank 120d)",
            color=COLORS["fbank"], alpha=0.85)
for bar, val in zip(b1, E008_EERS):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.15, f"{val:.1f}",
            ha="center", va="bottom", fontsize=8)
for bar, val in zip(b2, eers):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.15, f"{val:.1f}",
            ha="center", va="bottom", fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(["Fold 0", "Fold 1", "Fold 2"])
ax.set_ylabel("EER [%]")
ax.set_title("Per-fold EER — FBank vs E008")
ax.legend(fontsize=9)

# Mean ± std
ax = axes[1]
labels  = ["E008 +All\n(MFCC 39d)", "E016 +All\n(FBank 120d)"]
means_  = [np.mean(E008_EERS), mean_e]
stds_   = [np.std(E008_EERS),  std_e]
colors_ = [COLORS["e008"], COLORS["fbank"]]
bars_ = ax.bar(range(2), means_, color=colors_, alpha=0.85,
               yerr=stds_, capsize=8, error_kw=dict(elinewidth=2))
for bar, m, s in zip(bars_, means_, stds_):
    ax.text(bar.get_x() + bar.get_width()/2, m + s + 0.2,
            f"{m:.2f}\n±{s:.2f}", ha="center", fontsize=9)
best_idx_ = np.argmin(means_)
bars_[best_idx_].set_edgecolor("gold")
bars_[best_idx_].set_linewidth(3)
ax.annotate("★ best", xy=(best_idx_, means_[best_idx_] - stds_[best_idx_] - 0.3),
            ha="center", fontsize=9, color="goldenrod", fontweight="bold")
ax.set_xticks(range(2))
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel("EER mean ± std [%]")
ax.set_title("Mean ± std — E016 FBank vs E008 MFCC")

plt.suptitle("E016 — FBank 40+Δ+ΔΔ vs MFCC 13+Δ+ΔΔ", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
ticks     = [0.01, 0.05, 0.1, 0.2, 0.3, 0.4]
tick_pos  = [scipy_norm.ppf(t) for t in ticks]
tick_lbls = [f"{int(t*100)}" for t in ticks]

# Build E008 OOF reference from per-fold stored numbers (approximate DET)
# We plot E016 OOF DET and annotate E008 reference EER as a point
fig, ax = plt.subplots(figsize=(7, 6))

valid = ~np.isnan(oof_scores)
fpr, tpr, _ = roc_curve(y_all[valid], oof_scores[valid])
far_c = np.clip(fpr, 1e-4, 1-1e-4)
frr_c = np.clip(1-tpr, 1e-4, 1-1e-4)
ax.plot(scipy_norm.ppf(far_c), scipy_norm.ppf(frr_c),
        color=COLORS["fbank"], lw=2.5,
        label=f"E016 FBank 120d  OOF EER={eer_oof*100:.1f}%")

# E008 EER point on diagonal for reference
e008_eer_oof = 9.17 / 100
e008_pt = scipy_norm.ppf(e008_eer_oof)
ax.scatter([e008_pt], [e008_pt], s=80, color=COLORS["e008"], zorder=5,
           label=f"E008 MFCC 39d   OOF EER=9.17% (ref point)")

ax.plot(tick_pos, tick_pos, "k--", lw=1)
ax.set_xticks(tick_pos); ax.set_xticklabels(tick_lbls)
ax.set_yticks(tick_pos); ax.set_yticklabels(tick_lbls)
ax.set_xlabel("FAR [%]")
ax.set_ylabel("FRR [%]")
ax.set_title("DET curve — E016 FBank (OOF) vs E008 MFCC reference")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
fold_data = []
for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
    fold_data.append({
        "scores": oof_scores[val_idx],
        "labels": manifest.loc[val_idx, "label"].to_numpy()
    })

bin_edges = np.linspace(np.nanmin(oof_scores), np.nanmax(oof_scores), 35)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, (ax, fdata) in enumerate(zip(axes[:3], fold_data)):
    s, l = fdata["scores"], fdata["labels"]
    eer_f, thr_f = compute_eer(s[l==1], s[l==0])
    ax.hist(s[l==0], bins=bin_edges, alpha=0.6, color=COLORS["nontarget"],
            label="non-target", density=True)
    ax.hist(s[l==1], bins=bin_edges, alpha=0.75, color=COLORS["target"],
            label="target", density=True)
    ax.axvline(thr_f, color=COLORS["green"], ls="--", lw=2,
               label=f"thresh={thr_f:.2f}")
    ax.set_title(f"Fold {i}  —  EER={eer_f*100:.1f}%")
    ax.set_xlabel("Score (LLR)")
    ax.legend(fontsize=8)

ax = axes[3]
ax.hist(oof_scores[y_all==0], bins=bin_edges, alpha=0.6, color=COLORS["nontarget"],
        label="non-target", density=True)
ax.hist(oof_scores[y_all==1], bins=bin_edges, alpha=0.75, color=COLORS["target"],
        label="target", density=True)
ax.axvline(thr, color=COLORS["green"], ls="--", lw=2,
           label=f"OOF thresh={thr:.2f}")
ax.set_title(f"OOF overall  —  EER={eer_oof*100:.1f}%")
ax.set_xlabel("Score (LLR)")
ax.legend(fontsize=8)

plt.suptitle("E016 — Score distributions, FBank 40+Δ+ΔΔ", fontsize=12)
plt.tight_layout()
plt.show()

print(f"E008 +All (MFCC 39d):   EER={np.mean(E008_EERS):.2f}±{np.std(E008_EERS):.2f}%  (fold0=3.47, fold1=8.33, fold2=0.83)")
print(f"E016 +All (FBank 120d): EER={mean_e:.2f}±{std_e:.2f}%  (fold0={eers[0]:.2f}, fold1={eers[1]:.2f}, fold2={eers[2]:.2f})")
print(f"Delta: {mean_e - np.mean(E008_EERS):+.2f}% vs E008")